In [ ]:
# Phase 7 backbone family screen, fold 0: ConvNeXt-tiny and EVA02-small on c2's
# six-slot attention config, packed, judged by what each adds to c2+c3.
import glob, os, shutil, sys, time

GIT_SHA = '62e87a4'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.dataset import PreppedSlotDataset, _SLOT_PLANES
from knee.dicom import SLOTS, SLOTS_V3
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# The stale-src guard, derived from what this run needs. Without packing the
# arms would differ in BatchNorm statistics as well as the table; without the
# v3 plane lookup the second arm dies on its first study.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel._slot_features', KneeModel._slot_features, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__, ('weights_df',)),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, '_backbone_features'), 'rsna-knee-src predates slot packing'
assert all(name in _SLOT_PLANES for name, *_ in SLOTS_V3), 'rsna-knee-src cannot load v3 slots'
print('src carries slot packing and the v3 loader | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# The runs. Each family on its own recipe: a pretrained ViT at c2's OneCycle 3e-4
# would be destroyed (the reason c3 got differential rates), and a ConvNeXt is a
# CNN fine-tuned the way c2 was. The recipe is part of "use this backbone".
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = 'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,macro_auc,paired_delta,train_minutes,inference_seconds,promoted,gold_macro'
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

CONFIGS = [
    # name, backbone, (backbone_lr, head_lr) or None for c2's Adam/OneCycle
    ('f1_convnext_tiny_6slot_attn', 'convnext_tiny.in12k_ft_in1k', None),
    ('f2_eva02_small_6slot_attn', 'eva02_small_patch14_224.mim_in22k', (8e-6, 1e-3)),
]
HYPOTHESIS = {
    'f1_convnext_tiny_6slot_attn':
        'f1: ConvNeXt-tiny (in12k, apache-2.0, no BatchNorm) in place of efficientnet_b0 on '
        "c2's six-slot attention config and recipe. A CNN from a different pretraining set "
        'and normalisation, judged by its gain added to the c2+c3 fold-0 ensemble.',
    'f2_eva02_small_6slot_attn':
        'f2: EVA02-small (MIM in22k, MIT, no BatchNorm) on the same config with c3\'s '
        'differential AdamW recipe. A ViT pretrained differently from DINOv2, judged by its '
        'gain added to the c2+c3 fold-0 ensemble.',
}
SOURCE, FOLD = 'steven_v4', 0
N_GROUPS, OUT_SIZE, EPOCHS, BATCH, HEAD = 5, 224, 8, 8, 'slot_attention'
TIME_BUDGET_S = 5 * 3600
t_start = time.time()

def loader(uids, labels_df, shuffle=False, augment=False):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=N_GROUPS,
                            out_size=OUT_SIZE, slots=slot_names, augment=augment)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=2)

train_labels, eval_labels = LABEL_SOURCES[SOURCE]
tr_uids, va_uids = train_val_split(folds, val_fold=FOLD, exclude_uids=holdout)
tr_uids = [u for u in tr_uids if u in uid_to_npz]
va_uids = [u for u in va_uids if u in uid_to_npz]

results = []
for name, backbone, lrs in CONFIGS:
    if time.time() - t_start > TIME_BUDGET_S:
        print(f'time budget spent, skipping {name}')
        break
    print(f'\n=== {name}: {len(tr_uids)} train / {len(va_uids)} val, {len(slot_names)} slots x '
          f'{N_GROUPS} groups at {OUT_SIZE}px, {EPOCHS} epochs')

    train_loader = loader(tr_uids, train_labels, shuffle=True, augment=True)
    val_loader = loader(va_uids, eval_labels)
    model = KneeModel(backbone_name=backbone, num_labels=12, pretrained=True, head=HEAD).to(device)
    # Recomputes activations in the backward pass instead of storing them. Same
    # gradients; it is insurance against running out of T4 memory at six slots and
    # batch 8 without shrinking the batch, which would be a second variable
    # against c2. It costs compute, and these runs are loader-bound.
    model.backbone.set_grad_checkpointing(True)
    if lrs is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        max_lr = 3e-4
    else:
        optimizer = torch.optim.AdamW(differential_param_groups(model, *lrs), weight_decay=0.02)
        max_lr = [lrs[0] * 3, lrs[1] * 3]
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, epochs=EPOCHS, steps_per_epoch=len(train_loader))

    seen = []
    hook = model.backbone.register_forward_pre_hook(lambda _m, args: seen.append(args[0].shape[0]))
    t0 = time.time()
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, device=device,
                               scaler=scaler, scheduler=scheduler)
        if epoch == 0:
            hook.remove()
            print(f'  backbone images per batch {np.mean(seen):.1f} '
                  f'(unpacked {BATCH * len(slot_names) * N_GROUPS}), '
                  f'peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
        print(f'  epoch {epoch + 1}/{EPOCHS} loss {loss:.4f} ({(time.time() - t0) / 60:.1f} min)')
    minutes = (time.time() - t0) / 60

    y_true, y_pred = evaluate(model, val_loader, device=device)
    aucs = per_label_auc(y_true, y_pred)
    macro = macro_auc(y_true, y_pred)
    g_true, g_pred = evaluate(model, loader(gold_uids, gold_labels), device=device)
    gold_macro = macro_auc(g_true, g_pred)

    print(f'  {name}: val macro {macro:.4f}, GOLD macro {gold_macro:.4f} in {minutes:.1f} min')
    for label, auc in zip(LABEL_COLUMNS, aucs):
        print(f'    {label:20s} {auc:.4f}')
    results.append({'config': name, 'macro_auc': macro, 'gold_macro': gold_macro,
                    'minutes': minutes, 'final_loss': loss, 'n_val': len(va_uids)})
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name,
                   hypothesis=HYPOTHESIS[name], fold_set=f'primary_v3_fold{FOLD}',
                   seed=0, per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(macro), paired_delta=float('nan'),
                   train_minutes=minutes, inference_seconds=float('nan'), promoted=False,
                   gold_macro=float(gold_macro))
    np.save(f'/kaggle/working/oof_{name}.npy', y_pred)
    np.save(f'/kaggle/working/y_{name}.npy', y_true)
    np.save(f'/kaggle/working/gold_pred_{name}.npy', g_pred)
    np.save(f'/kaggle/working/gold_true_{name}.npy', g_true)
    torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

In [ ]:
# The judge: gain added to the c2+c3 pair, scored against the confirm runs' own
# fold-0 predictions -- the same studies, from the fold-0 models that are already
# in the submission. Probability mean is how Submission A blends; rank mean is
# printed beside it because a new family may be calibrated differently, which is
# when rank blending measured better before.
from knee.metrics import paired_macro_auc_delta

def mounted_one(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    assert len(hits) == 1, f'{pattern}: expected exactly one mounted file, found {hits}'
    return hits[0]

def confirm_fold(name):
    uids = pd.read_csv(mounted_one(f'oof_uids_{name}.csv'))['StudyInstanceUID'].astype(str)
    row = {u: i for i, u in enumerate(uids)}
    rows = [row[u] for u in va_uids]
    return (np.load(mounted_one(f'oof_true_{name}.npy'))[rows],
            np.load(mounted_one(f'oof_pred_{name}.npy'))[rows],
            np.load(mounted_one(f'gold_folds_{name}.npy'))[FOLD])

def rank_mean(*preds):
    return np.mean([pd.DataFrame(p).rank(pct=True).to_numpy() for p in preds], axis=0)

def same_labels(a, b):
    return np.array_equal(np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1))

c2_true, c2_val, c2_gold = confirm_fold('c2_6slot_attn')
_, c3_val, c3_gold = confirm_fold('c3_dinov2_2slot')

print(pd.DataFrame(results)[['config', 'macro_auc', 'gold_macro', 'final_loss', 'minutes']]
      .to_string(index=False))
for tier, truth_key, pred_key, c2p, c3p in (('val ', 'y', 'oof', c2_val, c3_val),
                                           ('gold', 'gold_true', 'gold_pred', c2_gold, c3_gold)):
    arms = {r['config']: np.load(f"/kaggle/working/{pred_key}_{r['config']}.npy") for r in results}
    if not arms:
        break
    y = np.load(f"/kaggle/working/{truth_key}_{results[0]['config']}.npy")
    if tier == 'val ':
        assert same_labels(y, c2_true), 'the confirm fold-0 rows are not the studies this run scored'
    pair = (c2p + c3p) / 2
    print(f'\n{tier} (n={len(y)}): c2 {macro_auc(y, c2p):.4f}, c3 {macro_auc(y, c3p):.4f}, '
          f'c2+c3 {macro_auc(y, pair):.4f}')
    for name, p in arms.items():
        d, lo, hi = paired_macro_auc_delta(y, c2p, p)
        print(f'  {name}: alone {macro_auc(y, p):.4f}, minus c2 {d:+.4f} [{lo:+.4f}, {hi:+.4f}]')
        d, lo, hi = paired_macro_auc_delta(y, pair, (c2p + c3p + p) / 3)
        verdict = 'CI excludes zero' if lo > 0 or hi < 0 else 'CI spans zero'
        print(f'    added to c2+c3, probability mean {d:+.4f} [{lo:+.4f}, {hi:+.4f}]  {verdict}')
        d, lo, hi = paired_macro_auc_delta(y, pair, rank_mean(c2p, c3p, p))
        print(f'    added to c2+c3, rank mean        {d:+.4f} [{lo:+.4f}, {hi:+.4f}]')
    if len(arms) == 2:
        threes = [(c2p + c3p + p) / 3 for p in arms.values()]
        better = max(threes, key=lambda e: macro_auc(y, e))
        d, lo, hi = paired_macro_auc_delta(y, better, (c2p + c3p + sum(arms.values())) / 4)
        print(f'  both families against the better three-member set: {d:+.4f} [{lo:+.4f}, {hi:+.4f}]')